# Lab 05-05 — Step-back: ask a broader question first, then merge

**Track 05 · Query transformation** — specific questions miss general context. "How does the parser handle overlap?" might only match the sentence that mentions overlap, while the passage that actually explains the splitter's design — the context the specific question depends on — never makes the top-k. Step-back prompting fixes this at the QUERY side: an LLM abstracts the question into a broader "step-back" question, the retriever runs BOTH the step-back and the original question, and the results are merged and deduplicated. The general context now has a retrieval pass of its own; the original query keeps precision.

This notebook is **self-contained**: it imports LangChain, sentence-transformers, and faiss directly — no repo component library. Every block of the pipeline is built right here:

```
user question ──► inline step-back prompt + ChatGroq ──► broader question ──► BGE embed ──► FAISS top-3 ──┐
user question ──► BGE embed ──► FAISS top-3 ───────────────────────────────────────────────────────────────┘
                                                                                    merge & dedupe ──► top-3
```

That is exactly how the shared components in `src/` work underneath: `src/retrieval/step_back.py` is this same prompt + fallback loop, and `src/llms/groq.py` is a thin wrapper around the same `ChatGroq`.

The lab compares, for the same questions:

* **RAW** — plain top-k: embed the question as the user typed it;
* **STEP-BACK** — merged retrieval over the step-back + original questions.

Same three questions as labs 01/02/04 (1606/1610/1626) so you can compare the transformations directly. The Groq LLM only writes the step-back question; embeddings stay local BGE.


## Setup

Two prerequisites must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` + `test.parquet`, already fetched by the repo's manifest-verified fetchers.
- **`GROQ_API_KEY` in the repo-root `.env`** — the Groq LLM writes the step-back question (it never embeds); embeddings stay local BGE. Without the key the step-back generation falls back to the raw question, exactly like the shared retriever's safety net.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-groq`, `sentence-transformers`, and `faiss-cpu`. The bootstrap cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu langchain-groq python-dotenv pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# LangChain + sentence-transformers + faiss — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_PASSAGES = 100` takes a deterministic head of the 3200-passage corpus; `QUESTION_IDS = [1606, 1610, 1626]` reuses labs 01/02/04's questions so the transformations are directly comparable; `TOP_K = 3` is the retrieval depth on both paths; `LLM_MODEL` names the Groq model that writes the step-back question (never the embedder); `BGE_MODEL_NAME` pins the local embedder. `PREVIEW` truncates the passage previews the demo prints next to each hit.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus (keeps runtime low)
QUESTION_IDS = [1606, 1610, 1626]  # same questions as labs 01/02/04, for comparison
TOP_K = 3
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq writes the step-back question, never embeds
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
PREVIEW = 62  # max characters of passage text shown next to each hit


## 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files

`load_passages` reads the first `n` passages (text + ids) from `passages.parquet`; `load_questions` pulls specific rows by id from `test.parquet`; `preview` flattens a passage onto one line for printing. Identical helpers to the other labs of this track keep the experiments directly comparable — the only thing that changes is the retriever wrapper.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — corpus + questions from the fresh rag-mini-wikipedia parquet files
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3. Experiment — raw vs step-back retrieval for the same questions

The whole pipeline is built inline. The embedder is `HuggingFaceEmbeddings` with the local BGE model (`normalize_embeddings=True`, which BGE requires for cosine); we embed the 100 passages once, then hand the FAISS store its vectors through a tiny precomputed passthrough — so the embed step and the index step stay separately timed, exactly like the lab. The inner retriever is a plain `similarity_search_by_vector` at `TOP_K = 3` (the inline shape of `src/retrieval/similarity.py`), and the step-back block is the same `STEPBACK_PROMPT` + `ChatGroq` the shared `StepBackRetriever` wraps, including its safety net: an empty or failed LLM output falls back to the original question. The merge is the same dedupe loop — step-back results first, then original-query results, first occurrence wins, cut to `TOP_K`.

The LLM section reports **run/skip** explicitly: with `GROQ_API_KEY` in `.env` it runs (`ChatGroq`, one call per question); without the key it prints SKIP and every step-back question falls back to the raw question — the same contract the lab's `GroqLLM` follows when the key is missing.


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — raw vs step-back retrieval for the same questions
# --------------------------------------------------------------------------
STEPBACK_PROMPT = """You are a search query abstractor for a RAG system.

Given a specific user question, produce ONE broader "step-back" question
that would retrieve the general background material the specific question
depends on.

Example:
  Specific: "How does the recursive character splitter set overlap?"
  Step-back: "How does the recursive character text splitter work?"

Rules:
- The step-back question must be broader, not a paraphrase.
- Keep the same language as the original question.
- Output only the step-back question, nothing else.

Specific question: {question}
Step-back question:"""


class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


class _ChatGroqLLM:
    """Inline stand-in for src/llms/groq.GroqLLM: ChatGroq + invoke(str) -> str."""

    def __init__(self, model: str, temperature: float = 0.0):
        self.model = model
        self.temperature = temperature
        self._llm = ChatGroq(model=model, temperature=temperature)

    def invoke(self, prompt: str) -> str:
        return self._llm.invoke(prompt).content


class _SimilarityRetriever:
    """Inline stand-in for src/retrieval/similarity.SimilarityRetriever."""

    def __init__(self, store, embedder, top_k: int = TOP_K):
        self.store = store
        self.embedder = embedder
        self.top_k = top_k

    def retrieve(self, question: str) -> list[Document]:
        """Embed the question and return the top-k most similar documents."""
        query_embedding = self.embedder.embed_query(question)
        return self.store.similarity_search_by_vector(query_embedding, k=self.top_k)


class _StepBackRetriever:
    """Inline stand-in for src/retrieval/step_back.StepBackRetriever."""

    def __init__(self, stepback_llm, retriever, top_k: int = TOP_K):
        self.stepback_llm = stepback_llm
        self.retriever = retriever
        self.top_k = top_k

    def _step_back(self, question: str) -> str:
        """Generate a broader step-back question, falling back to the original."""
        try:
            out = self.stepback_llm.invoke(
                STEPBACK_PROMPT.format(question=question)
            )
        except Exception:
            return question
        out = (out or "").strip()
        return out if out else question

    def retrieve(self, question: str) -> list[Document]:
        """Retrieve with step-back + original queries, dedupe, return top-k."""
        stepback = self._step_back(question)

        merged: list[Document] = []
        seen: set[str] = set()
        for doc in self.retriever.retrieve(stepback) + self.retriever.retrieve(question):
            key = doc.page_content
            if key and key not in seen:
                seen.add(key)
                merged.append(doc)
        return merged[: self.top_k]


def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)

    # --- Embed locally (BGE) and index in-memory ---------------------------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]
    t0 = time.perf_counter()
    store = FAISS.from_documents(chunks, embedding=_PrecomputedEmbeddings(passage_texts, passage_vecs))
    index_s = time.perf_counter() - t0

    # --- The two retrievers over the SAME store -----------------------------
    raw_retriever = _SimilarityRetriever(store, embedder, top_k=TOP_K)
    if os.getenv("GROQ_API_KEY"):
        stepback_llm = _ChatGroqLLM(model=LLM_MODEL)
        llm_status = f"run (ChatGroq {LLM_MODEL})"
    else:
        stepback_llm = None
        llm_status = ("skip (no GROQ_API_KEY in the repo-root .env — "
                      "step-back questions fall back to the raw question)")
        print("LLM section: SKIP —", llm_status)
    stepback_retriever = _StepBackRetriever(stepback_llm, raw_retriever, top_k=TOP_K)

    # --- Per question: raw retrieval + the step-back question + its merge ----
    results = []
    for qid, qtext in questions:
        raw_docs = raw_retriever.retrieve(qtext)
        t0 = time.perf_counter()
        stepback = stepback_retriever._step_back(qtext)
        stepback_s = time.perf_counter() - t0
        stepback_docs = stepback_retriever.retrieve(qtext)
        results.append(
            {
                "qid": qid,
                "question": qtext,
                "stepback": stepback,
                "stepback_s": stepback_s,
                "raw_docs": raw_docs,
                "stepback_docs": stepback_docs,
            }
        )

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "indexed": len(passage_texts),
        "embed_s": embed_s,
        "index_s": index_s,
        "llm_status": llm_status,
        "results": results,
    }


## 4. Demo — print the artifact

`print_demo(exp)` prints the artifact from three angles: the corpus subset with embedding/index timings and the LLM section's run/skip status; per question, the raw question, the step-back question the LLM wrote (with its generation time), and the top-1 passage of both paths; then a takeaway on why step-back works: the LLM abstracts the question into a broader, background-seeking version, so the general context the specific question depends on gets a retrieval pass of its own — at the cost of one LLM call + one extra embed + one extra FAISS query per question.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 05 — Step-back: ask a broader question first, then merge")
    print(f"{BGE_MODEL_NAME} (local) -> FAISS top-{TOP_K} -> {LLM_MODEL} step-back")
    print("=" * 66)

    print(f"\n[1] Corpus (deterministic subset, no randomness):")
    print(f"    {exp['indexed']} passages (first {N_PASSAGES} of 3200, ids {exp['passage_ids'][0]}..{exp['passage_ids'][-1]})")
    print(f"    embedded in {exp['embed_s']:.2f}s (dim 768), indexed in {exp['index_s']:.3f}s")
    print(f"    LLM section: {exp['llm_status']}")

    print(f"\n[2] Raw vs step-back (per question):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"]}] "{r["question"]}"')
        print(f"      step-back ({r['stepback_s']:.1f}s): {r['stepback']!r}")
        print(f"      raw        top-1: {preview(r['raw_docs'][0].page_content)}")
        print(f"      step-back  top-1: {preview(r['stepback_docs'][0].page_content)}")

    print("\n[3] Takeaway")
    print("    Step-back adds ONE broader retrieval pass for general context,")
    print("    then merges it with the precise original-query results (dupes")
    print("    dropped, first occurrence wins). The merge guarantees the")
    print("    answer's passage still surfaces while giving the background")
    print("    chunk a chance to enter the candidate set. Cost: one LLM call")
    print("    + one extra embed + one extra FAISS query per question.")


## 5. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: exactly `N_PASSAGES` passages indexed; every question returning `TOP_K` hits on both paths; the step-back question is non-empty and the merged results are deduplicated (step-back + original overlap); and the content checks — the step-back top-3 must still carry the answer's keyword (montevideo / spanish / 1930). The keyword checks are pinned to *retrieval outcomes*, not exact LLM wording, so the gate stays stable across runs. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural properties (no LLM involved).
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))
    checks.append(("each question returns TOP_K raw hits",
                   all(len(r["raw_docs"]) == TOP_K for r in exp["results"])))
    checks.append(("each question returns TOP_K step-back hits",
                   all(len(r["stepback_docs"]) == TOP_K for r in exp["results"])))

    # The step-back question must be non-empty.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        checks.append((f"{tag} step-back is non-empty",
                       bool(r["stepback"].strip())))

    # The merged results must be deduplicated (step-back + original overlap).
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        contents = [d.page_content for d in r["stepback_docs"]]
        checks.append((f"{tag} merged results are deduplicated",
                       len(contents) == len(set(contents))))

    # Content checks: the merged path must still surface the answer's keyword.
    # Q1606 -> Montevideo; Q1610 -> the Spanish; Q1626 -> 1930.
    for r in exp["results"]:
        tag = f"Q{r['qid']}"
        joined = " ".join(d.page_content for d in r["stepback_docs"]).lower()
        if r["qid"] == 1606:
            kw = "montevideo"
        elif r["qid"] == 1610:
            kw = "spanish"
        else:  # 1626
            kw = "1930"
        checks.append((f"{tag} step-back top-{TOP_K} retains '{kw}'", kw in joined))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A minute of embedding + 3 Groq step-back calls on the 100-passage subset — no downloads. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The three questions with the step-back question the LLM wrote, and the top-1 passage of the raw vs the merged step-back path. The step-back line shows the abstraction the LLM chose — a broader, background-seeking version of the question.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the parquet files are intact and the LLM section ran (see the status line in the demo).


In [ ]:
verify_gate(exp)
